# Notebook 03: Fine-Tuning Runs

**Goal:** Fine-tune on Python code (primary) and TinyStories prose (mandatory control).

**Outputs:** `checkpoints/code_seed*/`, `checkpoints/prose_seed*/`

**Runtime:** ~80 minutes per condition on Colab T4 GPU.

> **Note:** Both conditions are required. The prose control is mandatory for any domain-shift claim.

In [ ]:
import os
import sys
import numpy as np, torch, matplotlib.pyplot as plt
from pathlib import Path

# Define repository information
repo_name = "Mechanistic-Interpretability-Study-Induction-Circuit-Stability-Under-Fine-Tuning"
repo_url = "https://github.com/Mattral/Mechanistic-Interpretability-Study-Induction-Circuit-Stability-Under-Fine-Tuning"
repo_path = f"/content/{repo_name}"  # Standard Colab clone location

# Clone the repository if it doesn't exist
if not os.path.exists(repo_path):
    print(f"Cloning {repo_url} to {repo_path}...")
    !git clone {repo_url} {repo_path}

# Change current working directory to the repository root
# This allows relative imports (like 'src.model...') to work correctly
if os.getcwd() != repo_path:
    print(f"Changing current directory to {repo_path}")
    os.chdir(repo_path)

# Add the current directory (repo root) to sys.path if not already there
# This ensures 'src' is discoverable for imports.
if '.' not in sys.path:
    sys.path.insert(0, '.')

# Install project dependencies from requirements.txt
# This ensures all necessary libraries, including transformer_lens and transformers,
# are installed with the versions specified by the project.
print(f"Installing dependencies from {repo_path}/requirements.txt...")
!pip install -r requirements.txt

# Original imports
from src.model.config import ModelConfig, EvalConfig
from src.model.train import load_pretrained_model, set_global_seed
from src.circuits.patching import compute_circuit_attribution, get_circuit_heads
from src.viz.circuit_diagram import plot_circuit_diagram, plot_attribution_heatmap

set_global_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

### ⚠️ Restart Runtime Required
Please go to **Runtime -> Restart session** now. After the session restarts, run the cell below to load the modules and continue.

In [1]:
# After restarting the runtime, re-run this cell to continue with the imports and model setup.
import numpy as np, torch, matplotlib.pyplot as plt
from pathlib import Path

# The current working directory should already be set to the repo root from the previous cell.
# Add the current directory (repo root) to sys.path if not already there, in case of a fresh restart.
import sys
import os

repo_name = "Mechanistic-Interpretability-Study-Induction-Circuit-Stability-Under-Fine-Tuning"
repo_path = f"/content/{repo_name}"

if os.getcwd() != repo_path:
    print(f"Changing current directory to {repo_path}")
    os.chdir(repo_path)

if '.' not in sys.path:
    sys.path.insert(0, '.')

from src.model.config import ModelConfig, EvalConfig
from src.model.train import load_pretrained_model, set_global_seed
from src.circuits.patching import compute_circuit_attribution, get_circuit_heads
from src.viz.circuit_diagram import plot_circuit_diagram, plot_attribution_heatmap

set_global_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Changing current directory to /content/Mechanistic-Interpretability-Study-Induction-Circuit-Stability-Under-Fine-Tuning
Using device: cpu


In [2]:
import sys; sys.path.insert(0, '..')
import torch
from src.model.config import ModelConfig, TrainConfig
from src.model.finetune import run_finetuning
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
model_config = ModelConfig()

Device: cpu


In [ ]:
# --- Code fine-tuning (primary condition, 3 seeds) ---
for seed in [42, 123, 7]:
    print(f'\n=== Code, seed={seed} ===')
    cfg = TrainConfig(seed=seed, checkpoint_dir='../checkpoints', results_dir='../experiments/results')
    history = run_finetuning(model_config, cfg, run_name=f'code_seed{seed}', device=device, prose_control=False)
    print(f'Done: {len(history)} checkpoints')

2026-06-13T06:41:28 | INFO     | src.model.train | Global seed set to 42
2026-06-13T06:41:28 | INFO     | src.model.finetune | Fine-tuning run: code_seed42 | device: cpu
2026-06-13T06:41:28 | INFO     | src.model.train | Loading model 'attn-only-2l' on device 'cpu'



=== Code, seed=42 ===


config.json: 0.00B [00:00, ?B/s]

./model_final.pth:   0%|          | 0.00/210M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

2026-06-13T06:41:35 | INFO     | src.model.train | Model loaded: 2L 8H d_model=512


Loaded pretrained model attn-only-2l into HookedTransformer
Moving model to device:  cpu


2026-06-13T06:41:45 | INFO     | src.model.finetune | Baseline induction score mean: 0.0336
2026-06-13T06:43:01 | INFO     | src.circuits.patching | Circuit heads (>=0.5): [(1, 6)]
2026-06-13T06:43:01 | INFO     | src.model.finetune | Step-0 | IS_mean=0.0336 | task_loss=11.7863 | logit_diff_clean=4.8071
2026-06-13T06:43:01 | INFO     | src.model.train | Checkpoint saved step=0 -> ../checkpoints/code_seed42/step_000000.pt
2026-06-13T06:43:01 | INFO     | src.model.train | Global seed set to 42
2026-06-13T06:43:01 | INFO     | src.model.train | Training: 244 total steps, checkpoint every 100


Moving model to device:  cpu


In [ ]:
import os
from google.colab import files

# Define the base checkpoint directory
# Assuming current working directory is /content/Mechanistic-Interpretability-Study-Induction-Circuit-Stability-Under-Fine-Tuning
base_checkpoint_dir = os.path.join(os.getcwd(), 'checkpoints')

# Seeds used in the previous cell for code fine-tuning
seeds = [42, 123, 7]

print(f"Checking for checkpoints in: {base_checkpoint_dir}")

for seed in seeds:
    run_name = f'code_seed{seed}'
    checkpoint_path = os.path.join(base_checkpoint_dir, run_name)

    if os.path.exists(checkpoint_path) and os.path.isdir(checkpoint_path):
        print(f"Found checkpoint directory: {checkpoint_path}")
        zip_filename = f'{run_name}_checkpoints.zip'
        # Create a zip archive of the checkpoint directory
        !zip -r {zip_filename} {checkpoint_path} > /dev/null
        print(f"Downloading {zip_filename}...")
        files.download(zip_filename)
        print(f"Downloaded {zip_filename}.")
    else:
        print(f"Checkpoint directory not found for {run_name} at {checkpoint_path}. Skipping download.")

print("Finished checking and downloading code fine-tuning checkpoints.")

In [ ]:
# --- Prose control (mandatory, 3 seeds) ---
for seed in [42, 123, 7]:
    print(f'\n=== Prose, seed={seed} ===')
    cfg = TrainConfig(seed=seed, checkpoint_dir='../checkpoints', results_dir='../experiments/results')
    history = run_finetuning(model_config, cfg, run_name=f'prose_seed{seed}', device=device, prose_control=True)
    print(f'Done: {len(history)} checkpoints')

In [ ]:
import os
from google.colab import files

# Define the base checkpoint directory
# Assuming current working directory is /content/Mechanistic-Interpretability-Study-Induction-Circuit-Stability-Under-Fine-Tuning
base_checkpoint_dir = os.path.join(os.getcwd(), 'checkpoints')

# Seeds used in the previous cell for prose fine-tuning
seeds = [42, 123, 7]

print(f"Checking for checkpoints in: {base_checkpoint_dir}")

for seed in seeds:
    run_name = f'prose_seed{seed}' # Changed from 'code_seed' to 'prose_seed'
    checkpoint_path = os.path.join(base_checkpoint_dir, run_name)

    if os.path.exists(checkpoint_path) and os.path.isdir(checkpoint_path):
        print(f"Found checkpoint directory: {checkpoint_path}")
        zip_filename = f'{run_name}_checkpoints.zip'
        # Create a zip archive of the checkpoint directory
        !zip -r {zip_filename} {checkpoint_path} > /dev/null
        print(f"Downloading {zip_filename}...")
        files.download(zip_filename)
        print(f"Downloaded {zip_filename}.")
    else:
        print(f"Checkpoint directory not found for {run_name} at {checkpoint_path}. Skipping download.")

print("Finished checking and downloading prose fine-tuning checkpoints.")